In [28]:
import json

import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

In [29]:
annType = ['segm','bbox','keypoints']
annType = annType[2]      #specify type here
prefix = 'person_keypoints' if annType=='keypoints' else 'instances'
print('Running demo for *%s* results.'%(annType))

Running demo for *keypoints* results.


In [30]:
#initialize COCO ground truth api
dataDir='/home/dev/data/coco/'
dataType='val2017'
annFile = '%s/annotations/%s_%s.json'%(dataDir,prefix,dataType)
cocoGt=COCO(annFile)

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!


In [31]:
len(cocoGt.imgs)

5000

In [32]:
list(cocoGt.imgs.keys())[:10]

[397133, 37777, 252219, 87038, 174482, 403385, 6818, 480985, 458054, 331352]

# Making result file using ground truth data (AP of gt prediction data is lower than 1.0)

In [33]:
annFile

'/home/dev/data/coco//annotations/person_keypoints_val2017.json'

In [34]:
gt = json.load(open(annFile))

In [35]:
gt.keys()

dict_keys(['info', 'licenses', 'images', 'annotations', 'categories'])

In [37]:
gt['annotations'][0].keys()

dict_keys(['segmentation', 'num_keypoints', 'area', 'iscrowd', 'keypoints', 'image_id', 'bbox', 'category_id', 'id'])

In [22]:
gt_ret = []

for _ann in gt['annotations']:
    if _ann['category_id'] == 1 :
        gt_ret.append({'image_id': _ann['image_id'], 'category_id': 1, 'score': 1.0, 'keypoints': _ann['keypoints']})

gt_ret_file = "/home/dev/3rd-party/cocoapi/results/person_keypoints_val2017_gt.json"
with open(gt_ret_file, 'w') as fout:
    fout.write(json.dumps(gt_ret))

In [23]:
#initialize COCO detections api
cocoDt=cocoGt.loadRes(gt_ret_file)

Loading and preparing results...
DONE (t=0.16s)
creating index...
index created!


In [24]:
len(gt_ret)

11004

In [25]:
# running evaluation
cocoEval = COCOeval(cocoGt,cocoDt,annType)
# cocoEval.params.imgIds = list(cocoGt.imgs.keys())
cocoEval.params.imgIds = [_g['image_id'] for _g in gt_ret]
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=1.78s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.641
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.643
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.641
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 1.000
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 1.000


### 🤔🤔🤔
엥? AP가 1.0 이 안 나오네?

## 뭐가 문제인지 알았다.

하나씩 뒤져보니까, category_id==1 이면서 keypoint 값이 전부 0인 경우가 있네. 사람이 너무 작아서 레이블링 할 수 없는 경우다. 이걸 빼줘야 한다.

이렇게 하니까 AP 1.0 나오네.

In [27]:
gt_ret = []

for _ann in gt['annotations']:
    if _ann['category_id'] == 1 and sum(_ann['keypoints']) > 0:
        gt_ret.append({'image_id': _ann['image_id'], 'category_id': 1, 'score': 1.0, 'keypoints': _ann['keypoints']})

gt_ret_file = "/home/dev/3rd-party/cocoapi/results/person_keypoints_val2017_gt.json"
with open(gt_ret_file, 'w') as fout:
    fout.write(json.dumps(gt_ret))

#initialize COCO detections api
cocoDt=cocoGt.loadRes(gt_ret_file)

# running evaluation
cocoEval = COCOeval(cocoGt,cocoDt,annType)
# cocoEval.params.imgIds = list(cocoGt.imgs.keys())
cocoEval.params.imgIds = [_g['image_id'] for _g in gt_ret]
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

Loading and preparing results...
DONE (t=0.07s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.88s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 1.000
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDe

# Making result file using fake data

In [13]:
fake_kp = [
 36, 181, 2,
 20.25,191,1,
 35,166,2,
 20.25,191,1,
 8,171,2,
 20.25,191,1,
 2,246,2,
 20.25,191,1,
 20.25,191,1,
 20.25,191,1,
 20.25,191,1,
 20.25,191,0,
 20.25,191,0,
 20.25,191,0,
 20.25,191,0,
 20.25,191,0,
 20.25,191,0
]

In [14]:
fake_ret = []
for image_id in list(cocoGt.imgs.keys()):
    fake_ret.append({'image_id': image_id, 'category_id': 1, 'score': 0.8, 'keypoints': fake_kp})

In [15]:
fake_ret_file="/home/dev/3rd-party/cocoapi/results/person_keypoints_val2017_fakekeypoints_results.json"
with open(fake_ret_file, 'w') as fout:
    fout.write(json.dumps(fake_ret))

In [16]:
#initialize COCO detections api
cocoDt=cocoGt.loadRes(fake_ret_file)

Loading and preparing results...
DONE (t=0.06s)
creating index...
index created!


In [17]:
# running evaluation
cocoEval = COCOeval(cocoGt,cocoDt,annType)
cocoEval.params.imgIds = list(cocoGt.imgs.keys())
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.45s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.009
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.000
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.000
